In [ ]:
#TODO make sure this renders in the github repo

# Training an LLM

🌟 **LLM training occurs in two phases:**
1. **Pre-Training** (Phase 1):
   - Objective: (Teacher Forcing) Next-Token prediction over a massive, unstructured dataset (trillions of words from the internet, books, code).
   - Note: The pre-training phase is the most expensive part of training an LLM.
   - Goal: To teach the model the structure of human language, logic, coding syntax, and general world knowledge. This is the base model.
   - Steps: 
      1. Pass a input sequence of tokens through the model.
         - The tokens are masked so the model can not see future tokens, and specific to Llama, a document mask that prevents a token from one document from attending to tokens in other documents.
         - The ground truth is the same as the input, but shifted right by one token.
      2. The models outputs a prediction for every position at once, and we calculate cross-entropy loss between the model's prediction and the ground truth.

2. [**Post-Training** (Phase 2)](./post-training.ipynb)

---

**Llama 3 paper:**
  - "We start by converting a large, multilingual text corpus to discrete tokens and pre-training a large language model (LLM) on the resulting data to perform next-token prediction. In the language model pre-training stage, the model learns the structure of language and obtains large amounts of knowledge about the world from the text it is “reading”. To do this effectively, pre-training is performed at massive scale: we pre-train a model with $405B$ parameters on $15.6T$ tokens using a context window of $8K$ tokens. This standard pre-training stage is followed by a continued pre-training stage that increases the supported context window to $128K$ tokens. See Section 3 for details." 
  - "The recipe used to **pre-train Llama 3 405B** consists of three main stages: **(1) initial pre-training, (2) long-context pre-training, and (3) annealing**. The three stages are described separately below. We use similar recipes to pre-train the 8B and 70B models."
    - **Sub Pre-training stages:** (A single pre-training run)
       1. **Initial**: Most of pre-training is spent on this stage. Trains a massive, diverse mix of data (e.g., $15.6$ Trillion Tokens), using the standard learning rate schedule, and a **shorter context window** ($8{,}192$ tokens was used for the Llama 3). A shorter context window during this sub stage keeps the cost to train lower. Paper: "3.4.1 Initial Pre-Training ... We pre-train Llama 3 405B using AdamW ... Specifically, we use an initial batch size of 4M tokens and sequences of length 4,096, and double these values to a batch size of 8M sequences of 8,192 tokens after pre-training 252M tokens."
       2. **Long-Context:** Once the model has understand language from the initial stage, they **gradually increased the context window in six steps** from $8{,}192$ tokens to $128K$ tokens. They did this for the last ~$800$ Billion tokens on the run, because the model already knew how to predict words, it just needed to learn how to look further back in its memory to do it. Paper: "In Llama 3 405B pre-training, we increased context length gradually in six stages, starting from the original 8K context window and ending in the final 128K context window. This long-context pre-training stage was performed using approximately 800B training tokens."
       4. **Annealing:** Decayed the learning rate to $0$ past the `min_lr_ratio`, change the data mix to the absolute highest quality math and reasoning data sources. Paper source: "During pre-training on the final $40M$ tokens, we linearly annealed the learning rate to 0, maintaining a context length of $128K$ tokens. During this annealing phase, we also adjusted the data mix to upsample data sources of very high quality; see Section 3.1.3. Finally, we compute the average of model checkpoints (Polyak (1991) averaging) during annealing to produce the final pre-trained model."



**Notes:**
- Instead of training on a **number of epochs**, the LLMs are trained on a **number of steps**. Once it reaches the **total_training_steps** (e.g., 1,200,000), training ends. This is done because the datasets for LLM training are so massive, that the model will rarely see the same piece of text twice. 
- The dataloader yields dense, packed batches of tokens.

In [1]:
import EasyJupyter
import torch
from torch.utils.data import DataLoader
import torch.nn as nn

In [2]:
from typing import TYPE_CHECKING
if TYPE_CHECKING:
    from llama_configs import BaseConfig
    from model.decoder import Decoder

## Validation

Llama 3 paper: "Empirically, we find that annealing (see Section 3.4.3) on small amounts of high-quality code and mathematical data can boost the performance of pre-trained models on key benchmarks. Akin to Li et al. (2024b), we perform annealing with a data mix that upsamples high-quality data in select domains. We do not include any training sets from commonly used benchmarks in our annealing data. This enables us to assess the true few-shot learning capabilities and out-of-domain generalization of Llama 3. Following OpenAI (2023a), **we evaluate the efficacy of annealing on the GSM8k (Cobbe et al., 2021) and MATH (Hendrycks et al., 2021b) training sets in annealing.** We find that **annealing improved the performance** of a pre-trained Llama 3 8B model on the GSM8k and MATH validation sets by 24.0% and 6.4%, respectively. However, the improvements on the 405B model are negligible, suggesting that our flagship model has strong in-context learning and reasoning capabilities and does not require specific in-domain training samples to obtain strong performance."

## Loss Function

**Llama 3 Paper:** 
- "Language model **pre-training**. **We start by converting a large, multilingual text corpus to discrete tokens and pre-training a large language model (LLM) on the resulting data to perform next-token prediction.** **In the language model pre-training stage, the model learns the structure of language and obtains large amounts of knowledge about the world from the text it is “reading”.** To do this effectively, pre-training is performed at massive scale: we pre-train a model with 405B parameters on 15.6T tokens using a context window of 8K tokens. This standard pre-training stage is followed by a continued pre-training stage that increases the supported context window to 128K tokens. See Section 3 for details."


**Notes:**
- **Cross-Entropy Loss** was used for the **Pre-Training** (to calculate the error of next-token prediction compared to the ground truth) and in the SFT portion of **Post-Training**. **DPO Loss** is used in the alignment/Direct Preference Optimization portion of **Post-Training**.

## Pre-Training

[Optimizer and Scheduler](./utils/optimizer_and_scheduler.ipynb)

In [3]:
import time
from datetime import datetime, timedelta
from model.utils.model_io import save_checkpoint
from model.utils.plot import plot_loss_history
from model.utils.masking import create_causal_doc_mask
from typing import TYPE_CHECKING
from model.utils.data_loader import create_causal_dataloader

if TYPE_CHECKING:
    from torch.optim.lr_scheduler import LambdaLR
    from torch.utils.data import DataLoader
    import torch.optim as optim
    from model.model_text_only import TextOnlyModel


class PreTrainModel:
     

    def __init__(
        self,
        cfg: BaseConfig,
        model: TextOnlyModel,
        optimizer: optim.AdamW,
        scheduler: LambdaLR,
    ):
        """
        Implement the pre-training.
        """
        self.cfg = cfg
        self.model = model

        self.criterion = nn.CrossEntropyLoss(
            # label_smoothing=0.1,  # Used Torch's built-in Label Smoothing.
            ignore_index=cfg.special_tokens["pad_token"]["ID"],
            # reduction="sum",
        )
        self.optimizer = optimizer
        self.scheduler = scheduler

        # Track the training steps. Once it reach `total_training_steps` training ends.
        self.step_counter = 0
        self.loss_history = []
        self.start_time = None  # Track the start time of the training

        # Exponential Moving Average printing for step time
        self.alpha = 0.1
        self.ema_step_time = None
        self.last_print_time = None

        # During the long-context stage, we gradually increase the context length in six steps
        self.increase_context_len_steps_tracker = 0
        self.initial_duration = int(cfg.pre_training_stages_steps["initial_perc"] * cfg.total_training_steps)
        self.lc_duration = int(cfg.pre_training_stages_steps["long_context_perc"] * cfg.total_training_steps)
        self.lc_start_step = self.initial_duration
        self.annealing_start_step = self.lc_start_step + self.lc_duration

        self.cfg.training_stage = "pre-trained"

        

    def train(
        self,
        tokenizer,
        continue_from_step=0,
        overfit=False,
        print_every: int = 100,
    ):
        """
        Pre-train the model.

        Args:
            continue_from_step: If continuing training, start from the last step the model was trained on.
            train_dataloader: A dataloader that yields packed batches of tokens.
            overfit: Whether to do a small overfit test.
            print_every: Print every n steps.
        """

        # TODO Add a validation loop to validate while training Maybe take a slice of the coming dataset and set it to the validation set every n steps and then that n steps run the validation loop, maybe set n steps to be the same as save config interval
        self.step_counter = continue_from_step

        # Set the model to train mode
        self.model.train()

        # Convert the dataloader stream into an iterator for step-based fetching
        dataloader = create_causal_dataloader(
            self.cfg, tokenizer, ds=self.cfg.pre_training_ds["initial_and_lc_stage"]
        )  
        data_iter = iter(dataloader)

        grad_accum_steps = self._setup(overfit, data_iter)
        self.start_time = time.time()
        self.last_print_time = self.start_time

        while self.step_counter < self.cfg.total_training_steps:

            # Handle sub-stage transitions -> Pre-training stages: (1) Initial, (2) Long-Context, (3) Annealing ===
            grad_accum_steps, train_dataloader, data_iter = self._handle_stage_transitions(grad_accum_steps, train_dataloader, data_iter)

            # NOTE: If you are training on massive GPU clusters, use HuggingFace's Accelerator for gradient accumulation!
            accumulated_loss = 0.0

            # === Inner loop: Accumulate gradients
            for micro_step in range(grad_accum_steps):
                if overfit:
                    x_ids, y_ids = static_x, static_y
                else:
                    try:
                        # Fetch exactly one micro-batch from the huggingFace stream
                        x_ids, y_ids = next(data_iter)
                    except StopIteration:
                        # Fallback in case a finite dataset exhausts
                        data_iter = iter(train_dataloader)
                        x_ids, y_ids = next(data_iter)

                x_ids = x_ids.to(self.cfg.device)
                y_ids = y_ids.to(self.cfg.device)

                # Run micro-batch, and do not call optimizer.step() there!
                loss = self._run_micro_batch(x_ids, y_ids, grad_accum_steps)
                accumulated_loss += loss

            # === Outer loop: The global step
            # Clip gradients to prevent exploding gradients
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)

            # Apply the accumulated gradients to the weights
            self.optimizer.step()
            if not overfit:
                self.scheduler.step()
            self.optimizer.zero_grad()

            avg_loss = accumulated_loss / grad_accum_steps
            self.loss_history.append(avg_loss)
            self.step_counter += 1

            self._print_step_info(
                avg_loss=avg_loss,
                print_every=print_every,
            )

            if self.step_counter % 200 == 0 and not overfit:
                # Save a checkpoint every 200 steps
                self._save_checkpoint(avg_loss)

        self._save_checkpoint(avg_loss)
        print("\n\nTraining complete!\n\n")

    def _run_micro_batch(self, x_ids, y_ids, grad_accum_steps):
        """
        Run a single forward and backward pass for a micro-batch.

        Args:
            x_ids: The input token ids.
            y_ids: The output token ids.
        """

        # Generated the doc and causal mask
        batch_mask = create_causal_doc_mask(cfg=self.cfg, token_ids=x_ids)

        # Forward pass
        logits = self.model(x_ids, batch_mask)

        # Compute the loss
        loss = self.criterion(
            logits.view(-1, self.cfg.vocab_size), y_ids.view(-1)
        )  # Flatten to 2D and 1D for CrossEntropy

        # Scale the loss to account for gradient accumulation
        scaled_loss = loss / grad_accum_steps

        # Backward pass
        scaled_loss.backward()

        # Return the unscaled loss
        return loss.item()

    def _print_step_info(self, avg_loss, print_every):
        if self.step_counter % print_every == 0:
            curr_time = time.time()

            # Calculate time taken for just the last `print_every` steps
            internal_time = curr_time - self.last_print_time
            actual_step_time = internal_time / print_every

            # Update Exponential Moving Average
            if self.ema_step_time is None:
                self.ema_step_time = actual_step_time
            else:
                self.ema_step_time = (self.alpha * actual_step_time) + (
                    1 - self.alpha
                ) * self.ema_step_time

            remaining_steps = self.cfg.total_training_steps - self.step_counter
            eta_seconds = self.ema_step_time * remaining_steps

            eta_str = str(timedelta(seconds=int(eta_seconds)))
            elapsed_str = str(timedelta(seconds=int(curr_time - self.start_time)))

            print(
                f"[{datetime.now().strftime('%m-%d %H:%M:%S')}] | "
                f"Step [{self.step_counter}/{self.cfg.total_training_steps}] | "
                f"Avg Step: {self.ema_step_time:.2f} | "
                f"Elapsed: {elapsed_str} | "
                f"ETA: {eta_str} | "
                f"Loss: {avg_loss:.4f} "
            )
            self.last_print_time = curr_time

    def _setup(self, overfit, data_iter):
        """
        Setups up the environment for the training, either for overfitting or for normal training. Returns the gradient accumulation steps.
        """
        self.model.train()

        if overfit:
            print("\n\nOverfitting the model on a single batch......")
            global static_x, static_y
            static_x, static_y = next(data_iter)

            grad_accum_steps = 1
            # Override the 0.0 warmup learning rate, this is because we skip self.scheduler.step() during overfit, so learning rate never increments.
            for param_group in self.optimizer.param_groups:
                param_group["lr"] = self.cfg.peak_lr
            return 1  # Temporarily override gradient accumulation to 1 for direct updates, needed for overfitting test

        grad_accum_steps = self.cfg.gradient_accumulation_steps
        print("\n" + "#" * 64)
        print(
            f"\nPre-Training Model | Total Training Steps: {self.cfg.total_training_steps:,} | "
            f"Accumulation Steps: {grad_accum_steps}"
        )
        print("\n" + "#" * 64)
        return self.cfg.gradient_accumulation_steps

    def _handle_stage_transitions(self, grad_accum_steps, train_dataloader, data_iter):
        """
        Handle sub-stage transitions -> Pre-training stages: (1) Initial, (2) Long-Context, (3) Annealing
        """

        # # --- Initial Stage ---
        # if self.step_counter < self.initial_steps:
        #     pass
        
        # --- Long-Context Stage ---
        if self.lc_start_step <= self.step_counter < self.annealing_start_step:
            steps_in_lc = self.step_counter - self.lc_start_step

            # For the first 6 steps gradually increase the context length
            inverse_size = max(1, self.lc_duration//6)
            if steps_in_lc % inverse_size == 0:
                stage_idx = steps_in_lc // inverse_size

                if stage_idx < 6:
                    increase_per_step = (self.cfg.max_context_len - self.cfg.initial_context_len) // 6 
                    new_len = self.cfg.initial_context_len + (increase_per_step * (stage_idx + 1))
                    self.cfg.update_context_len(new_len)

                    grad_accum_steps = self.cfg.gradient_accumulation_steps
                    print(f"\n[Long-Context Stage {stage_idx+1}/6] context_len: {self.cfg.context_len} | grad_accum:     {grad_accum_steps}")
                    train_dataloader, data_iter = self._refresh_dataloader(tokenizer=train_dataloader.dataset.tokenizer, ds=self.cfg.pre_training_ds["initial_and_lc_stage"])

        
        # --- Annealing Stage ---
        elif self.step_counter == self.annealing_start_step:
            # The scheduler will automatically handle the learning rate decay to 0.
            print("\nTransition to Annealing stage (Linear Decay to 0, and swapping to a high quality dataset).\n")
            train_dataloader, data_iter = self._refresh_dataloader(tokenizer=train_dataloader.dataset.tokenizer, ds=self.cfg.pre_training_ds["annealing_stage"])

            
        return grad_accum_steps, train_dataloader, data_iter



    def _save_checkpoint(self, avg_loss):
        save_checkpoint(
            cfg=self.cfg,
            model=self.model,
            optimizer=self.optimizer,
            scheduler=self.scheduler,
            step_counter=self.step_counter,
            loss_history=self.loss_history,
            avg_loss=avg_loss,
        )

        self.cfg.save_config_to_json()
        plot_loss_history(
            cfg=self.cfg, loss_history=self.loss_history, step_counter=self.step_counter
        )


    def _refresh_dataloader(self, tokenizer, ds:dict):
        """Re-instantiate the dataloader with the updated context length or swap datasets."""
        new_dataloader = create_causal_dataloader(cfg=self.cfg, tokenizer=tokenizer, ds=ds)
        return new_dataloader, iter(new_dataloader)

/Users/tonyavis/miniconda3/envs/how_to_build_an_llm_env/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## OVERFIT TEST:

In [ ]:
# @i-c
from llama_configs import Llama3_scaled_down
from model.model_text_only import TextOnlyModel
from model.tokenizer import BPETokenizer
from model.utils.data_loader import create_causal_dataloader
from model.utils.data_loader import get_raw_dataset
from model.utils.optimizer_and_scheduler import get_optimizer, get_scheduler

print("\n ---------- OVERFIT TEST ---------- \n")
cfg = Llama3_scaled_down()

# Temporarily override gradient accumulation to 1 for direct updates, needed for overfitting test
cfg.gradient_accumulation_steps = 1
cfg.token_budget = 250 * cfg.global_batch_size_tokens

cfg.CURR_CHPT_DIR = cfg.CHPTS_DIR / "Scaled_down_Llama_3_1_OVERFIT"
cfg.CURR_CHPT_DIR.mkdir(parents=True, exist_ok=True)


tokenizer = BPETokenizer(cfg)
success, trained_tokenizer = tokenizer.load_tokenizer()
if not success:
    trained_tokenizer = tokenizer.train_tokenizer(get_raw_dataset(ds=cfg.pre_training_ds["initial_and_lc_stage"]))

model = TextOnlyModel(cfg).to(cfg.device)

optimizer = get_optimizer(cfg, model=model)
scheduler = get_scheduler(cfg, optimizer=optimizer)

trainer = PreTrainModel(cfg, model=model, optimizer=optimizer, scheduler=scheduler)


 ---------- OVERFIT TEST ---------- 


Project Root: /Users/tonyavis/Main/How_to_build_an_LLM
Training the BPE tokenizer from stream | Max Document Count: 500000 | chunk_size: 1000...



In [ ]:
# @i-c:
# Fetch a single batch
print("\n\nFetching a single batch...")
dataloader = create_causal_dataloader(cfg, trained_tokenizer)

Took about **~4 mins** to run the overfit test on my M1 Mac Max with 64GB RAM.

In [ ]:
# @i-c: 
trainer.train(dataloader, overfit=True, print_every=25)
print("If the loss approaches 0.0, the model architecture and gradient flow are functional.")

# CANDEL

In [ ]:
# @i-c
from llama_configs import Llama3_scaled_down
from model.model_text_only import TextOnlyModel
from model.tokenizer import BPETokenizer
from model.utils.data_loader import create_causal_dataloader
from model.utils.data_loader import get_raw_dataset
from model.utils.optimizer_and_scheduler import get_optimizer, get_scheduler

cfg = Llama3_scaled_down()
model = TextOnlyModel(cfg).to(cfg.device)
optimizer = get_optimizer(cfg, model=model)
scheduler = get_scheduler(cfg, optimizer=optimizer)

i = PreTrainModel(cfg=cfg, model=model, optimizer=optimizer, scheduler=scheduler)

In [ ]:
i.initial_steps, i.long_context_steps, i.annealing_steps